# Ordenamiento Territorial

> **Contexto de dominio:** [`docs/fuentes/ordenamiento_territorial.md`](../../docs/fuentes/ordenamiento_territorial.md)  
> **Bloque:** A | **Línea:** `ordenamiento_territorial`  
> **Variable principal:** `superficie_km2` (km²)  
> **Modelos sugeridos:** RANDOM_FOREST, XGBOOST  
> Flujo: `Plan.md` sección 3 — ciclo estadístico completo.

**Antes de comenzar:** Leer `docs/fuentes/ordenamiento_territorial.md` para entender:
- Variables ambientales clave y sus rangos físicos
- Normativa colombiana aplicable (umbrales normativos)
- Fuentes de datos oficiales y frecuencia de actualización
- Preguntas analíticas típicas de esta línea

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "ordenamiento_territorial"
VARIABLE = "superficie_km2"
UNIDAD = "km²"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `ordenamiento_territorial` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "ordenamiento_territorial.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos
> Colocar el archivo en `data/raw/` y ajustar la ruta.  
> Ver sección **Datos y fuentes** de `docs/fuentes/ordenamiento_territorial.md` para las fuentes oficiales.

In [ ]:
# df = load_csv("data/raw/ordenamiento_territorial.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "superficie_km2": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

<!-- ENRICHMENT: ordenamiento_territorial -->

## 1b. Estructura Ecologica Principal (EEP) e indicadores de presion territorial

La **EEP** es el conjunto de areas y ecosistemas estrategicos que sostienen los procesos ecologicos del territorio. Es la base para la zonificacion ambiental en el POT (Ley 388/1997).

| Componente EEP | Norma | Funcion |
|---|---|---|
| Paramos | Ley 1930/2018 | Regulacion hidrica, provision agua |
| Humedales | Ley 357/1997 (Ramsar) | Regulacion hidrica, biodiversidad |
| Rondas hidricas | Res. 957/2018 | Proteccion cauces, zonas de inundacion |
| Areas Protegidas | Decreto 1076/2015 | Conservacion biodiversidad |

**Indicadores de presion:**
- **TCCN (Tasa de Cambio de Coberturas Naturales):** perdida anual en % de ecosistemas estrategicos
- **IACAL (Indice de Alteracion Potencial de la Calidad del Agua):** presion por vertimientos
- **IACAL formula:** Cargas_contaminantes_vertidas / Oferta_hidrica_disponible

In [ ]:
# EEP sintetica + TCCN + IACAL por municipio
np.random.seed(55)
n = len(df)

# Coberturas EEP (ha) -- tendencia decreciente por expansion urbana/agricola
componentes_eep = ['Paramo', 'Humedal', 'Bosque_Andino', 'Ronda_Hidrica']
n_muni = 12
municipios = [f'Mpio-{i+1:02d}' for i in range(n_muni)]
ha_eep = {
    comp: np.random.uniform(100, 5000, n_muni)
    for comp in componentes_eep
}
df_eep = pd.DataFrame({'municipio': municipios, **ha_eep})
df_eep['total_eep_ha'] = df_eep[componentes_eep].sum(axis=1)

# TCCN: tasa de cambio de coberturas naturales (%/ano)
df_eep['tccn_pct'] = np.random.uniform(-3.5, -0.2, n_muni).round(2)  # negativo = perdida

# IACAL: cargas vertimientos / oferta hidrica (0-1 bajo; >2 critico)
carga_dbo_ton = np.random.uniform(50, 800, n_muni)  # ton DBO/ano
oferta_hm3 = np.random.uniform(5, 200, n_muni)     # hm3/ano
df_eep['iacal'] = (carga_dbo_ton / oferta_hm3).round(3)

def iacal_cat(v):
    if v < 0.1: return 'Muy bajo'
    if v < 1.0: return 'Bajo'
    if v < 2.0: return 'Medio'
    if v < 5.0: return 'Alto'
    return 'Muy alto'

df_eep['iacal_cat'] = df_eep['iacal'].apply(iacal_cat)

from estadistica_ambiental.config import IUA_THRESHOLDS
print('IUA_THRESHOLDS (referencia para IACAL):', IUA_THRESHOLDS)
print(f'\nIACAL | min={df_eep["iacal"].min():.3f} max={df_eep["iacal"].max():.3f}')
print(f'Municipios IACAL Alto/Muy Alto: {(df_eep["iacal"] >= 2).sum()}/{n_muni}')
print(f'TCCN promedio: {df_eep["tccn_pct"].mean():.2f}%/ano (negativo = perdida coberturas)')
df_eep[['municipio','total_eep_ha','tccn_pct','iacal','iacal_cat']]

## 2. Validación y EDA
> `validate()` usa rangos físicos específicos para `ordenamiento_territorial` desde `config.py`.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_ordenamiento_territorial.html",
       title="EDA — Ordenamiento Territorial", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

In [ ]:
plot_series(df, "fecha", "superficie_km2", title="Ordenamiento Territorial — superficie_km2 (km²)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "superficie_km2", period="month")
plt.show()

## 3c. IACAL y Estructura Ecologica Principal — Dashboard territorial

La econometria espacial con **modelos SARAR** (Spatial AutoRegressive with AutoRegressive disturbances) permite modelar como la presion territorial en un municipio afecta a sus vecinos:

```
y = rho * W * y + X * beta + u      (SAR: autocorrelacion en y)
u = lambda * W * u + e              (AR: autocorrelacion en errores)
```

Donde W = matriz de contigüedad espacial (reina/torre). Implementacion en Python: `spreg.GM_Combo` de `pysal/spreg`.

> Un IACAL alto en un municipio genera externalidades hidricas negativas aguas abajo (efecto autocorrelacion espacial positiva). Esto justifica el enfoque cuencal del POMCA.

In [ ]:
IACAL_COLORS = {'Muy bajo':'#27ae60','Bajo':'#82e0aa','Medio':'#f1c40f',
                'Alto':'#e67e22','Muy alto':'#e74c3c'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: IACAL por municipio
ax = axes[0]
bar_colors = df_eep['iacal_cat'].map(IACAL_COLORS)
ax.barh(df_eep['municipio'], df_eep['iacal'], color=bar_colors, alpha=0.85)
ax.axvline(2.0, color='red', ls='--', lw=1.5, label='Umbral alto IACAL=2')
ax.set_title('IACAL — Alteracion Potencial Calidad Agua', fontweight='bold')
ax.set_xlabel('IACAL (carga/oferta)'); ax.legend(fontsize=7); ax.grid(axis='x', alpha=0.3)

# Panel B: Composicion EEP por municipio
ax = axes[1]
bottom = np.zeros(n_muni)
colors_eep = ['#1a73e8','#27ae60','#82e0aa','#f1c40f']
for comp, color in zip(componentes_eep, colors_eep):
    ax.bar(range(n_muni), df_eep[comp], bottom=bottom, color=color, alpha=0.85, label=comp)
    bottom += df_eep[comp].values
ax.set_xticks(range(n_muni)); ax.set_xticklabels(df_eep['municipio'], rotation=45, ha='right', fontsize=7)
ax.set_title('Composicion EEP por Municipio (ha)', fontweight='bold')
ax.set_ylabel('Hectareas'); ax.legend(fontsize=7); ax.grid(axis='y', alpha=0.3)

# Panel C: TCCN vs IACAL
ax = axes[2]
sc = ax.scatter(df_eep['tccn_pct'], df_eep['iacal'],
                c=df_eep['iacal_cat'].map(IACAL_COLORS), s=80, alpha=0.8, edgecolors='gray')
ax.axhline(2.0, color='red', ls='--', lw=1)
ax.axvline(-2.0, color='orange', ls='--', lw=1)
ax.set_xlabel('TCCN (%/ano)'); ax.set_ylabel('IACAL')
ax.set_title('Presion territorial: TCCN vs IACAL\n(SARAR: efecto espacial sobre cuenca)', fontweight='bold')
for _, r in df_eep.iterrows():
    ax.annotate(r['municipio'], (r['tccn_pct'], r['iacal']), fontsize=6)
ax.grid(alpha=0.3)

plt.suptitle('EEP + IACAL + TCCN — Determinantes ambientales POT (Ley 388/1997)',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show()

print('Modelo SARAR (spreg.GM_Combo) requiere: pip install spreg')
print(f'Municipios EEP < 500 ha total: {(df_eep["total_eep_ha"] < 500).sum()} — prioridad restauracion')

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial
> ADR-004: Estacionariedad obligatoria pre-ARIMA (ADF + KPSS juntos).

In [ ]:
ts = df.set_index("fecha")["superficie_km2"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas
> Compara `superficie_km2` contra las normas colombianas relevantes.

In [ ]:
rep = exceedance_report(df["superficie_km2"], variable="superficie_km2")
if rep.empty:
    print("Sin normas colombianas registradas para 'superficie_km2'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["superficie_km2"], method="linear")
print(f"Faltantes antes: {df["superficie_km2"].isna().sum()} | "
      f"después: {df_clean["superficie_km2"].isna().sum()}")

## 7. Modelos predictivos

In [ ]:
ts = df_clean.set_index("fecha")["superficie_km2"]

models = {
    "RandomForest": get_model("random_forest", lags=[1,2,3,6,12]),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 7b. Guardrails y supuestos metodológicos
<!-- GUARDRAILS: ordenamiento_territorial -->

> **Antes de publicar resultados**, verificar que se cumplen los supuestos clave del flujo. Esta sección lista los más comunes y los específicos de la línea.

### Supuestos comunes (todas las líneas)

- **Estacionariedad (ADR-004):** ADF + KPSS deben coincidir antes de aplicar ARIMA. Si discrepan, diferenciar conservadoramente o usar modelos no-ARIMA (Prophet, ML).
- **Outliers (ADR-002):** los picos ambientales son señal real (eventos, episodios). No aplicar clipping automático — sólo `preprocessing/outliers.py` opt-in y documentado.
- **Métrica primaria (ADR-003):** RMSLE NO en variables que pueden ser negativas o cero. Usar MAE + sMAPE (o NSE / KGE en hidrología) como default.
- **Tamaño muestral mínimo:** ARIMA requiere ≥ 36 observaciones; STL anual con datos diarios, ≥ 2 ciclos completos.
- **Residuos (post-fit):** verificar normalidad (Jarque-Bera) e independencia (Ljung-Box, lag = 12). Residuos correlacionados → modelo subespecificado.
- **Walk-forward con gap (BP-1):** series con ACF ≥ 0.7 inflan R². Usar `gap ≥ horizonte` en `walk_forward()`.
- **Normas oficiales:** usar `config.NORMA_*` y `config.*_THRESHOLDS` — nunca umbrales hardcodeados en el notebook (ADR-005).

### Supuestos específicos — Ordenamiento territorial

- **Largo plazo (POT/POMCA):** horizonte 12 años; tendencias decadales > variabilidad anual.
- **Lag ENSO = 6 meses** (respuesta lenta del territorio a forzamiento climático).
- **Escalas espaciales:** veredal → municipal → departamental — no mezclar.

### Antes de presentar a la autoridad ambiental

- Reportar intervalos de confianza, no sólo el punto estimado.
- Documentar el período de los datos, los gaps y el método de imputación usado.
- Registrar decisiones metodológicas no triviales en `docs/decisiones.md` (ADRs).


## 8. Conclusiones

- **Línea temática:** Ordenamiento Territorial (`ordenamiento_territorial`)
- **Variable analizada:** `superficie_km2` (km²)
- **Modelos ejecutados:** RANDOM_FOREST, XGBOOST
- Completar con hallazgos reales al trabajar con datos de producción.

### Normativa y referencias
- Ver `docs/fuentes/ordenamiento_territorial.md` para normativa colombiana, indicadores oficiales y fuentes de datos.
- Registrar decisiones metodológicas en `docs/decisiones.md`.